# Orchestrator-Workers — Oil & Gas Well Performance Investigation

## Pattern
The orchestrator-workers pattern has a **coordinator LLM that analyses an input and determines at runtime which subtasks to create**, then delegates each to a specialized worker LLM. Unlike parallelization, the subtasks are not predefined — they emerge from the data.

This is the right pattern when:
- The optimal analysis depends on the specific input — you can't predefine the subtasks
- Different problems require different combinations of investigative angles
- A single-prompt approach would miss context-dependent nuance

## O&G Use Case: Well Performance Investigation
When an engineer asks "Why is Well X underperforming?", the right investigation depends entirely on what the data shows:
- Falling tubing pressure + high water cut → liquid loading + water source investigation
- Rising GOR + declining rate → gas coning or reservoir depletion analysis
- Pressure OK but rate declining → choke/skin/damage investigation
- New well underperforming → completions and stimulation review

The orchestrator reads the well data, decides which of these apply, and spins up only the relevant workers. A reservoir engineer wouldn't run every test on every well — they'd look at the data and choose. This pattern does the same.

In [ ]:
%pip install openai -q

In [ ]:
import sys
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sys.path.insert(0, '/Workspace' + _nb_path.rsplit('/', 1)[0])
except Exception:
    sys.path.insert(0, '.')

from util import llm_call, extract_xml

def parse_investigations(xml_text: str) -> list[dict]:
    """Parse orchestrator XML output into a list of investigation tasks."""
    import re
    investigations = []
    for block in re.findall(r'<investigation>(.*?)</investigation>', xml_text, re.DOTALL):
        inv = {}
        for field in ['type', 'hypothesis', 'worker_prompt']:
            m = re.search(f'<{field}>(.*?)</{field}>', block, re.DOTALL)
            inv[field] = m.group(1).strip() if m else ''
        if inv.get('type'):
            investigations.append(inv)
    return investigations


def investigate_well(well_id: str, well_data: str) -> dict:
    """Run a dynamic well performance investigation.

    The orchestrator determines which analyses are warranted based on
    the specific well data, then delegates each to a worker LLM.

    Args:
        well_id: Well identifier.
        well_data: Well performance data string.

    Returns:
        Dict with orchestrator analysis, investigations run, and synthesis.
    """
    # ── Step 1: Orchestrator analyses the data and plans the investigation ──
    orchestrator_prompt = f"""You are a senior reservoir engineer reviewing a well performance concern.

Analyse the well data below and determine which investigations are warranted.
Choose ONLY the investigations that the data actually supports — do not run every possible test.
Typical investigations (select 2-4 as appropriate):
  - decline_curve: Is the rate decline steeper than expected for this reservoir?
  - liquid_loading: Is the well loading up with liquids? Check pressure/rate correlation.
  - water_source: What is causing elevated water cut? Coning, channeling, or integrity issue?
  - gas_coning: Is GOR rising? Is gas breakthrough from the cap occurring?
  - skin_damage: Could near-wellbore damage be restricting flow? Check pressure drop vs. rate.
  - choke_optimization: Is the choke position optimal for current reservoir conditions?
  - completions_review: For newer wells, are the perforations/stimulation delivering expected results?
  - artificial_lift: Would artificial lift (ESP, gas lift, plunger) recover deferred production?

Respond in XML:
<orchestrator_assessment>
  2-3 sentence summary of what the data shows and why you chose these investigations.
</orchestrator_assessment>
<investigations>
  <investigation>
    <type>investigation_type_key</type>
    <hypothesis>What you expect to find and why the data suggests it.</hypothesis>
    <worker_prompt>Specific analytical instructions for this investigation given the data.</worker_prompt>
  </investigation>
  ... (2-4 investigations only)
</investigations>

Well ID: {well_id}
Well Data:
{well_data}"""

    print(f"\n{'='*60}")
    print(f"ORCHESTRATOR — Analysing {well_id}")
    print('='*60)

    orchestrator_response = llm_call(orchestrator_prompt)
    assessment = extract_xml(orchestrator_response, 'orchestrator_assessment')
    investigations = parse_investigations(orchestrator_response)

    print(f"Assessment: {assessment.strip()}")
    print(f"Investigations planned: {[i['type'] for i in investigations]}")

    # ── Step 2: Workers execute each investigation ──
    worker_results = []
    for inv in investigations:
        print(f"\n  → Running worker: {inv['type']}")
        worker_prompt = f"""You are a petroleum engineer conducting a focused well investigation.

Investigation type: {inv['type']}
Hypothesis to test: {inv['hypothesis']}

Instructions: {inv['worker_prompt']}

Well Data:
{well_data}

Provide a concise technical finding (3-5 sentences). State whether the hypothesis is
CONFIRMED / PARTIALLY CONFIRMED / NOT SUPPORTED by the data, and give your specific evidence.
End with one concrete recommended action."""

        finding = llm_call(worker_prompt)
        worker_results.append({'type': inv['type'], 'hypothesis': inv['hypothesis'], 'finding': finding})
        print(f"    ✓ {inv['type']} complete")

    # ── Step 3: Synthesise all findings into a single recommendation ──
    findings_text = '\n\n'.join([f"{r['type'].upper()}:\n{r['finding']}" for r in worker_results])
    synthesis_prompt = f"""You are a senior reservoir engineer. Synthesise the following investigation
findings for Well {well_id} into a single actionable recommendation for the asset manager.

Format:
ROOT CAUSE ASSESSMENT:
  [What is actually causing the underperformance — primary and contributing factors]

RECOMMENDED INTERVENTION:
  [Specific action, in priority order, with expected production uplift if estimable]

MONITORING PLAN:
  [What to track after intervention to confirm it worked]

Findings:
{findings_text}"""

    print(f"\n  → Synthesising findings...")
    synthesis = llm_call(synthesis_prompt)

    return {
        'well_id': well_id,
        'assessment': assessment,
        'investigations': worker_results,
        'synthesis': synthesis
    }

## Mock Well Data — Three Different Underperformance Scenarios
Each well has a different root cause. The orchestrator should plan different investigations for each.

In [ ]:
WELLS = [
    {
        "id": "Well C-22",
        "data": """
        Field: Permian Basin Block 7 West
        Reservoir: Wolfcamp A, depth 9,200 ft TVD
        Production history: On production 18 months

        Current performance (last 7 days avg):
          Oil rate: 198 bbl/d  (IP was 520 bbl/d, target 300 bbl/d)
          Gas rate: 0.31 MMscfd
          Water cut: 29% (was 8% at 6 months)
          Tubing pressure: 410 psi (was 890 psi last week, 1,100 psi at 6 months)
          Casing pressure: 1,750 psi (stable)
          Casing-tubing differential: 1,340 psi
          Choke: 32/64" (unchanged for 3 months)

        Trend: Tubing pressure dropped 54% in one week. Rate declined 35% concurrent.
        GOR: 1,565 scf/bbl (up from 1,200 scf/bbl at 6 months)
        Operator note: Well sounds slugging. Liquid surges at surface every 20-30 min.
        Artificial lift: None installed
        """
    },
    {
        "id": "Well A-17",
        "data": """
        Field: Permian Basin Block 7 West
        Reservoir: Wolfcamp B, depth 9,800 ft TVD
        Production history: On production 36 months

        Current performance (last 7 days avg):
          Oil rate: 258 bbl/d  (IP was 380 bbl/d, target 280 bbl/d)
          Gas rate: 0.42 MMscfd
          Water cut: 67% (was 22% at 12 months, 45% at 24 months)
          Tubing pressure: 620 psi (stable)
          Casing pressure: 890 psi (stable)
          Choke: 48/64" (widened from 32/64" 4 months ago to maintain rate)

        Trend: Water cut has been accelerating for 12 months. Rate maintained by progressively
               opening choke but gross fluid rate increasing.
        GOR: 1,628 scf/bbl (gradual increase, up from 1,400 at 12 months)
        Offset well data: A-18 (100m away, same zone) showing 48% water cut
        Completion: 40-stage frac, 2,800 lb/ft proppant loading
        Operator note: Choke is already at 48/64, limited room to compensate further.
        """
    },
    {
        "id": "Well B-09",
        "data": """
        Field: Permian Basin Block 7 West
        Reservoir: Wolfcamp A, depth 9,150 ft TVD
        Production history: On production 6 months (new well)

        Current performance (last 7 days avg):
          Oil rate: 512 bbl/d  (IP was 680 bbl/d, target 500 bbl/d — ON TARGET)
          Gas rate: 1.1 MMscfd
          Water cut: 4% (stable and low)
          Tubing pressure: 3,210 psi
          Casing pressure: 3,180 psi
          Choke: 16/64" (tight choke, constrained)

        GOR: 2,148 scf/bbl — SIGNIFICANTLY above Wolfcamp A field norm of 1,400 scf/bbl
        GOR trend: 1,750 at month 1, 1,950 at month 3, 2,148 at month 6 — rising
        Offset well B-08 (same zone, 150m away): GOR 1,380 scf/bbl, stable
        Completion: 38-stage frac, 2,600 lb/ft proppant — standard for this zone
        Pressures: High and near-equal tubing/casing pressures suggest strong reservoir energy
        Operator note: Meeting rate target but GOR trend is unusual for a new Wolfcamp well.
        """
    }
]

print(f"{len(WELLS)} wells queued for investigation")

## Run Orchestrated Well Investigations

In [ ]:
all_results = []
for well in WELLS:
    result = investigate_well(well['id'], well['data'])
    all_results.append(result)

## Synthesis — Final Recommendations Per Well

In [ ]:
for r in all_results:
    print(f"\n{'='*60}")
    print(f"  {r['well_id']} — INVESTIGATION SUMMARY")
    print('='*60)
    print(f"\nOrchestrator ran: {[i['type'] for i in r['investigations']]}")
    print(f"\n{r['synthesis']}")

## Why the Orchestrator Matters

The three wells above should produce different investigation plans:

| Well | Expected investigations | Why |
|---|---|---|
| C-22 | liquid_loading, artificial_lift | Sudden pressure drop + slugging = classic liquid loading |
| A-17 | water_source, choke_optimization | Accelerating water cut, choke already wide open |
| B-09 | gas_coning, decline_curve | Rising GOR on a new well with strong pressures |

If you ran a fixed set of 5 investigations on every well, you'd waste LLM calls on irrelevant analyses and dilute the output. The orchestrator ensures each well gets the investigation it actually needs — just like a reservoir engineer would decide which tests to run based on the symptoms.

**Where to go from here:**
- Connect to live production data in Unity Catalog Delta tables
- Add offset well comparison by querying a nearby-wells dataset
- Output recommendations to a work order system or Slack channel
- Run weekly via DABs job across all underperforming wells in the portfolio